# **SpaceX Falcon 9 First Stage Landing Prediction**


## Data Visualization and Feature Engineering


In this notebook, I'll continue my analysis to predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is due to the fact that SpaceX can reuse the first stage.

This notebook focuses on exploratory data visualization and feature engineering to prepare for machine learning model development.


## Objectives

My goals in this notebook are to:

* Perform visual exploratory data analysis using `Pandas`, `Matplotlib`, and `Seaborn`
* Prepare the data through feature engineering for future machine learning applications


### Import Libraries and Define Auxiliary Functions


First, I'll import the necessary libraries for data analysis and visualization


In [ ]:
# pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Matplotlib is a plotting library for python and pyplot gives us a MatLab like plotting framework. I'll use this to plot the data.
import matplotlib.pyplot as plt
# Seaborn is a Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics
import seaborn as sns

## Exploratory Data Analysis


First, I'll read the SpaceX dataset I created earlier into a Pandas dataframe and examine its contents


In [ ]:
df = pd.read_csv("data/dataset_part_2.csv")
df.head(5)

I'll begin by examining how `FlightNumber` (indicating the continuous launch attempts) and `Payload` variables might affect the landing outcome.

By plotting `FlightNumber` vs. `PayloadMass` and overlaying the outcome of the launch, I can observe if there are any patterns. Based on my initial hypothesis, as the flight number increases, the first stage might be more likely to land successfully. The payload mass should also be important; with more massive payloads, I'd expect the first stage to have a lower chance of successful return.


In [ ]:
# Create a catplot to visualize Flight Number vs Payload Mass, colored by landing outcome
sns.catplot(y="PayloadMass", x="FlightNumber", hue="Class", data=df, aspect=5)
plt.xlabel("Flight Number", fontsize=20)
plt.ylabel("Payload Mass (kg)", fontsize=20)
plt.show()

From my analysis of the data, I've observed that different launch sites have different success rates. `CCAFS LC-40` has a success rate of approximately 60%, while `KSC LC-39A` and `VAFB SLC 4E` have higher success rates of around 77%.


Next, I'll analyze each launch site in detail to visualize their specific launch records.


In [ ]:
# Visualizing the relationship between Flight Number and Launch Site
sns.catplot(x="FlightNumber", y="LaunchSite", hue="Class", data=df, aspect=3)
plt.xlabel("Flight Number", fontsize=20)
plt.ylabel("Launch Site", fontsize=20)
plt.show()

Examining the Flight Number vs. Launch Site plots, I can observe several patterns. Early flights (lower flight numbers) show more failures across all launch sites. As SpaceX gained experience with higher flight numbers, the success rate generally improved. This suggests that the company's landing technology and procedures have evolved and improved over time.


In [ ]:
# Visualizing the relationship between Payload Mass and Launch Site
sns.catplot(x="PayloadMass", y="LaunchSite", hue="Class", data=df, aspect=3)
plt.xlabel("Payload Mass (kg)")
plt.ylabel("Launch Site")
plt.show()

I'm also interested in examining if there's any relationship between launch sites and their payload mass capacity.


From the Payload vs. Launch Site scatter plot, I can observe that the VAFB-SLC launch site does not handle very heavy payloads (greater than 10,000 kg). This is likely due to its geographic location or facility capabilities. The other sites appear to accommodate a wider range of payload masses.


In [ ]:
# Visualizing the success rate of each orbit type
df_success_rate = df.groupby("Orbit")["Class"].mean().reset_index()
df_success_rate.plot(kind="bar", x="Orbit", y="Class")

Next, I want to check if there's any relationship between success rate and orbit type.


The bar chart shows varying success rates across different orbit types. Some orbits appear to have higher success rates than others, which could be due to factors like the required velocity, trajectory, and fuel consumption for each orbit type. This information will be valuable for my prediction model.


In [ ]:
# Visualizing the relationship between Flight Number and Orbit type
sns.catplot(x="FlightNumber", y="Orbit", hue="Class", data=df, aspect=3)
plt.xlabel("Flight Number")
plt.ylabel("Orbit Type")
plt.show()

For each orbit type, I want to see if there's any relationship between FlightNumber and the success of missions to that orbit.


From this visualization, I can see that in the LEO (Low Earth Orbit), the success rate appears to be related to the flight number - more recent flights show higher success rates. However, for GTO (Geostationary Transfer Orbit), there seems to be no clear relationship between flight number and success rate. This suggests that GTO missions might have inherent challenges regardless of SpaceX's experience level.


In [ ]:
# Visualizing the relationship between Payload Mass and Orbit type
sns.catplot(x="PayloadMass", y="Orbit", hue="Class", data=df, aspect=3)
plt.xlabel("Payload Mass (kg)")
plt.ylabel("Orbit Type")
plt.show()

Similarly, I'll plot the Payload vs. Orbit charts to reveal any relationship between payload mass and orbit type success rates.


This visualization shows that for heavier payloads, the successful landing rates are higher for Polar, LEO, and ISS orbits. However, for GTO, the pattern is less clear as both successful and unsuccessful landings occur across various payload masses. This suggests that GTO missions might be more challenging regardless of payload mass, possibly due to the higher energy requirements for this orbit type.


Now I'll examine the trend of success rate over time by plotting the average success rate by year.


First, I'll create a function to extract the year from the date field:


In [ ]:
# A function to Extract years from the date 
year = []
def Extract_year():
    for i in df["Date"]:
        year.append(i.split("-")[0])
    return year
Extract_year()
df['Date'] = year
df.head()

In [ ]:
# Plot a line chart showing the success rate by year
success_rate = df.groupby("Date")["Class"].mean().reset_index()
plt.plot(success_rate["Date"], success_rate["Class"])
plt.xlabel("Year")
plt.ylabel("Success Rate")

From this visualization, I can observe that the success rate has generally increased from 2013 to 2020. This trend demonstrates SpaceX's improving technology and experience with first stage landings over time.


## Features Engineering


Based on my exploratory analysis, I now have some preliminary insights about how various factors affect the success rate of first stage landings. Now I'll select and prepare the features that will be used for building a prediction model in future notebooks.


In [ ]:
# Select the features to be used in the prediction model
features = df[['FlightNumber', 'PayloadMass', 'Orbit', 'LaunchSite', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial']]
features.head()

Many machine learning algorithms require numerical input, but several of my features are categorical. I'll use one-hot encoding to convert categorical variables like `Orbit`, `LaunchSite`, `LandingPad`, and `Serial` into numerical features.


In [ ]:
# Apply one-hot encoding to categorical columns
features_one_hot = pd.get_dummies(features, columns=["Orbit", "LaunchSite", "LandingPad", "Serial"])
features_one_hot.head()

Now that my `features_one_hot` dataframe contains only numerical values, I'll convert all columns to the same data type (`float64`) for consistency.


In [ ]:
# Convert all features to float64 type
features_one_hot = features_one_hot.astype("float64")

Finally, I'll export this processed dataset for use in my machine learning models in the next notebook.


In [ ]:
# Export the preprocessed features for later use
#features_one_hot.to_csv("data/dataset_part_3.csv", index=False)